In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import accuracy_score, f1_score

In [4]:
import numpy as np

data = np.load("../models/preprocessed_data.npz")

X_train_early_processed = data["X_train_early"]
X_val_early_processed = data["X_val_early"]

y_train_encoded = data["y_train"]
y_val_encoded = data["y_val"]

In [5]:
def build_dnn(input_features, learning_rate=0.001, neurons1=64, neurons2=32):

    model = Sequential([
        Dense(neurons1, activation='relu', input_shape=(input_features,)),
        Dropout(0.3),

        Dense(neurons2, activation='relu'),
        Dropout(0.2),

        Dense(4, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [6]:
configs = [
    {"name": "Original", "lr": 0.001, "n1": 64, "n2": 32},
    {"name": "Lower Learning Rate", "lr": 0.0005, "n1": 64, "n2": 32},
    {"name": "More Neurons", "lr": 0.001, "n1": 128, "n2": 64}
]

results = []

for config in configs:

    model = build_dnn(
        input_features=X_train_early_processed.shape[1],
        learning_rate=config["lr"],
        neurons1=config["n1"],
        neurons2=config["n2"]
    )

    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    )

    history = model.fit(
        X_train_early_processed,
        y_train_encoded,
        validation_data=(X_val_early_processed, y_val_encoded),
        epochs=100,
        batch_size=16,
        callbacks=[early_stopping],
        verbose=0
    )

    predictions = np.argmax(
        model.predict(X_val_early_processed, verbose=0),
        axis=1
    )

    accuracy = accuracy_score(y_val_encoded, predictions)
    macro_f1 = f1_score(y_val_encoded, predictions, average='macro')

    results.append({
        "Configuration": config["name"],
        "Accuracy": accuracy,
        "Macro F1": macro_f1
    })

results_df = pd.DataFrame(results)
results_df

c:\Users\Parv\Desktop\DNN Project\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\Parv\Desktop\DNN Project\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\Parv\Desktop\DNN Project\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(act

,Configuration,Accuracy,Macro F1
0,Original,0.556701,0.440254
1,Lower Learning Rate,0.536082,0.370385
2,More Neurons,0.505155,0.384606


In [7]:
best_config = results_df.loc[results_df["Macro F1"].idxmax()]

print("Best Configuration:")
print(best_config)

Best Configuration:
Configuration    Original
Accuracy         0.556701
Macro F1         0.440254
Name: 0, dtype: object
